# Retail & E-Commerce Business Analytics

## Data Cleaning & Data Quality Assessment

This notebook focuses on validating data quality, converting data types, checking relationships between tables, and creating business-ready features for analysis.

In [61]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [62]:
customers = pd.read_csv("C:\\Users\\Mayuri\\Downloads\\Olist_Dataset\\olist_customers_dataset.csv")
orders = pd.read_csv("C:\\Users\\Mayuri\\Downloads\\Olist_Dataset\\olist_orders_dataset.csv")
order_items = pd.read_csv("C:\\Users\\Mayuri\\Downloads\\Olist_Dataset\\olist_order_items_dataset.csv")
payments = pd.read_csv("C:\\Users\\Mayuri\\Downloads\\Olist_Dataset\\olist_order_payments_dataset.csv")
reviews = pd.read_csv("C:\\Users\\Mayuri\\Downloads\\Olist_Dataset\\olist_order_reviews_dataset.csv")
products = pd.read_csv("C:\\Users\\Mayuri\\Downloads\\Olist_Dataset\\olist_products_dataset.csv")
sellers = pd.read_csv("C:\\Users\\Mayuri\\Downloads\\Olist_Dataset\\olist_sellers_dataset.csv")
geolocation = pd.read_csv("C:\\Users\\Mayuri\\Downloads\\Olist_Dataset\\olist_geolocation_dataset.csv")
category = pd.read_csv("C:\\Users\\Mayuri\\Downloads\\Olist_Dataset\\olist_product_category.csv")

In [63]:
# Verify Data Loading

print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Payments:", payments.shape)
print("Reviews:", reviews.shape)
print("Products:", products.shape)
print("Sellers:", sellers.shape)
print("Geolocation:", geolocation.shape)
print("Category:", category.shape)

Customers: (99441, 5)
Orders: (99441, 8)
Order Items: (112650, 7)
Payments: (103886, 5)
Reviews: (99224, 7)
Products: (32951, 9)
Sellers: (3095, 4)
Geolocation: (1000163, 5)
Category: (71, 2)


In [64]:
# Convert Date Columns
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column])
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


In [65]:
# Primary Key Validation
primary_keys = {
    "Customers": ("customer_id", customers),
    "Orders": ("order_id", orders),
    "Products": ("product_id", products),
    "Sellers": ("seller_id", sellers)
}

for table, (key, df) in primary_keys.items():
    print(f"\n{table}")
    print(f"Total Rows : {len(df)}")
    print(f"Unique IDs : {df[key].nunique()}")

    if len(df) == df[key].nunique():
        print("Primary Key is Unique")
    else:
        print("Duplicate Primary Keys Found")


Customers
Total Rows : 99441
Unique IDs : 99441
Primary Key is Unique

Orders
Total Rows : 99441
Unique IDs : 99441
Primary Key is Unique

Products
Total Rows : 32951
Unique IDs : 32951
Primary Key is Unique

Sellers
Total Rows : 3095
Unique IDs : 3095
Primary Key is Unique


In [66]:
# Orders should have valid Customer IDs

missing_customers = orders[
    ~orders["customer_id"].isin(customers["customer_id"])
]

print("Missing Customer IDs:", len(missing_customers))

Missing Customer IDs: 0


In [67]:
missing_orders = payments[
    ~payments["order_id"].isin(orders["order_id"])
]
print("Missing Order IDs in Payments:", len(missing_orders))

Missing Order IDs in Payments: 0


In [68]:
missing_reviews = reviews[
    ~reviews["order_id"].isin(orders["order_id"])
]

print("Missing Order IDs in Reviews:", len(missing_reviews))

Missing Order IDs in Reviews: 0


In [69]:
missing_products = order_items[
    ~order_items["product_id"].isin(products["product_id"])
]

print("Missing Product IDs:", len(missing_products))

Missing Product IDs: 0


In [70]:
missing_sellers = order_items[
    ~order_items["seller_id"].isin(sellers["seller_id"])
]

print("Missing Seller IDs:", len(missing_sellers))

Missing Seller IDs: 0


In [71]:
orders.isnull().sum()/len(orders)*100

order_id                         0.000000
customer_id                      0.000000
order_status                     0.000000
order_purchase_timestamp         0.000000
order_approved_at                0.160899
order_delivered_carrier_date     1.793023
order_delivered_customer_date    2.981668
order_estimated_delivery_date    0.000000
dtype: float64

In [72]:
# Create a new column to calculate the number of days taken to deliver each order
orders["delivery_time_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.days

In [73]:
# Create a new column to calculate delivery delay (positive = late, negative = early)
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.days

In [74]:
# Display the newly created business columns

orders[[
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
    "order_delivered_customer_date",
    "delivery_time_days",
    "delivery_delay_days"
]].head()

,order_purchase_timestamp,order_estimated_delivery_date,order_delivered_customer_date,delivery_time_days,delivery_delay_days
0,2017-10-02 10:56:33,2017-10-18,2017-10-10 21:25:13,8.0,-8.0
1,2018-07-24 20:41:37,2018-08-13,2018-08-07 15:27:45,13.0,-6.0
2,2018-08-08 08:38:49,2018-09-04,2018-08-17 18:06:29,9.0,-18.0
3,2017-11-18 19:28:06,2017-12-15,2017-12-02 00:28:42,13.0,-13.0
4,2018-02-13 21:18:39,2018-02-26,2018-02-16 18:17:02,2.0,-10.0
